# Q2.1 — Baseline vs Fine-Tuned Transformer

Compare classical baselines (TF-IDF + LR/SVM) against a fine-tuned DistilBERT on **Sentiment** and **Sarcasm**.

**Evaluation metric:** Macro-F1 (primary), plus per-class precision and recall.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {DEVICE}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

Device: mps


## 1 — Load & Prepare Data

In [2]:
dataset = load_dataset("surrey-nlp/BESSTIE-CW-26")

train_df = dataset["train"].to_pandas()
val_df   = dataset["validation"].to_pandas()
test_df  = dataset["test"].to_pandas()

# Labels as int
for df in [train_df, val_df, test_df]:
    df["Sentiment"] = df["Sentiment"].astype(int)
    df["Sarcasm"] = df["Sarcasm"].astype(int)

X_train, y_train_sent, y_train_sarc = train_df["text"], train_df["Sentiment"], train_df["Sarcasm"]
X_val,   y_val_sent,   y_val_sarc   = val_df["text"],   val_df["Sentiment"],   val_df["Sarcasm"]
X_test,  y_test_sent,  y_test_sarc  = test_df["text"],  test_df["Sentiment"],  test_df["Sarcasm"]

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Sentiment balance (train): {y_train_sent.value_counts().to_dict()}")
print(f"Sarcasm balance (train):    {y_train_sarc.value_counts().to_dict()}")

Train: 3747 | Val: 313 | Test: 2183
Sentiment balance (train): {0: 1907, 1: 1840}
Sarcasm balance (train):    {0: 3223, 1: 524}


## 2 — Classical Baselines (TF-IDF + LR / SVM)

In [3]:
def train_evaluate_baseline(model_cls, model_name, X_tr, y_tr, X_va, y_va, X_te, y_te, task_name):
    """Train a TF-IDF + classifier pipeline and return metrics."""
    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
    X_tr_vec = vectorizer.fit_transform(X_tr)
    X_va_vec  = vectorizer.transform(X_va)
    X_te_vec  = vectorizer.transform(X_te)

    clf = model_cls(max_iter=2000, random_state=SEED)
    if isinstance(clf, LogisticRegression):
        clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)
    elif isinstance(clf, LinearSVC):
        clf = LinearSVC(max_iter=2000, class_weight="balanced", random_state=SEED)

    clf.fit(X_tr_vec, y_tr)
    y_pred = clf.predict(X_te_vec)

    macro_f1 = f1_score(y_te, y_pred, average="macro")
    precision = precision_score(y_te, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_te, y_pred, average="macro", zero_division=0)

    print(f"\n{'='*60}")
    print(f"{model_name} — {task_name}")
    print(f"{'='*60}")
    print(f"Macro-F1:   {macro_f1:.4f}")
    print(f"Macro-Prec: {precision:.4f}")
    print(f"Macro-Rec:  {recall:.4f}")
    print(classification_report(y_te, y_pred, digits=4))

    return {
        "model": model_name,
        "task": task_name,
        "macro_f1": macro_f1,
        "precision": precision,
        "recall": recall,
        "y_pred": y_pred,
        "vectorizer": vectorizer,
        "clf": clf,
    }

# --- Sentiment ---
baseline_results = []
baseline_results.append(train_evaluate_baseline(
    LogisticRegression, "TF-IDF + LR", X_train, y_train_sent, X_val, y_val_sent, X_test, y_test_sent, "Sentiment"
))
baseline_results.append(train_evaluate_baseline(
    LinearSVC, "TF-IDF + SVM", X_train, y_train_sent, X_val, y_val_sent, X_test, y_test_sent, "Sentiment"
))

# --- Sarcasm ---
baseline_results.append(train_evaluate_baseline(
    LogisticRegression, "TF-IDF + LR", X_train, y_train_sarc, X_val, y_val_sarc, X_test, y_test_sarc, "Sarcasm"
))
baseline_results.append(train_evaluate_baseline(
    LinearSVC, "TF-IDF + SVM", X_train, y_train_sarc, X_val, y_val_sarc, X_test, y_test_sarc, "Sarcasm"
))


TF-IDF + LR — Sentiment
Macro-F1:   0.8347
Macro-Prec: 0.8362
Macro-Rec:  0.8344
              precision    recall  f1-score   support

           0     0.8221    0.8648    0.8429      1117
           1     0.8502    0.8039    0.8264      1066

    accuracy                         0.8351      2183
   macro avg     0.8362    0.8344    0.8347      2183
weighted avg     0.8358    0.8351    0.8349      2183




TF-IDF + SVM — Sentiment
Macro-F1:   0.8371
Macro-Prec: 0.8378
Macro-Rec:  0.8369
              precision    recall  f1-score   support

           0     0.8307    0.8568    0.8435      1117
           1     0.8448    0.8171    0.8307      1066

    accuracy                         0.8374      2183
   macro avg     0.8378    0.8369    0.8371      2183
weighted avg     0.8376    0.8374    0.8373      2183




TF-IDF + LR — Sarcasm
Macro-F1:   0.6255
Macro-Prec: 0.6129
Macro-Rec:  0.6747
              precision    recall  f1-score   support

           0     0.9161    0.8019    0.8552      1878
           1     0.3098    0.5475    0.3957       305

    accuracy                         0.7664      2183
   macro avg     0.6129    0.6747    0.6255      2183
weighted avg     0.8314    0.7664    0.7910      2183




TF-IDF + SVM — Sarcasm
Macro-F1:   0.6065
Macro-Prec: 0.6055
Macro-Rec:  0.6075
              precision    recall  f1-score   support

           0     0.8904    0.8871    0.8888      1878
           1     0.3205    0.3279    0.3241       305

    accuracy                         0.8090      2183
   macro avg     0.6055    0.6075    0.6065      2183
weighted avg     0.8108    0.8090    0.8099      2183



## 3 — Fine-Tuned DistilBERT

In [4]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from datasets import Dataset

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

def compute_metrics(eval_pred):
    from sklearn.metrics import f1_score, precision_score, recall_score
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "precision": precision_score(labels, preds, average="macro", zero_division=0),
        "recall": recall_score(labels, preds, average="macro", zero_division=0),
    }

print(f"Using model: {MODEL_NAME}")
print(f"Device: {DEVICE}")

Using model: distilbert-base-uncased
Device: mps


In [5]:
def train_transformer(task_name, label_col, seed=SEED):
    """Fine-tune DistilBERT on a given task. Returns model, predictions, and metrics."""
    print(f"\n{'#'*60}")
    print(f"# Training DistilBERT for: {task_name} (seed={seed})")
    print(f"{'#'*60}")

    # Prepare datasets
    train_ds = Dataset.from_pandas(train_df[["text", label_col]].rename(columns={label_col: "label"}))
    val_ds   = Dataset.from_pandas(val_df[["text", label_col]].rename(columns={label_col: "label"}))
    test_ds  = Dataset.from_pandas(test_df[["text", label_col]].rename(columns={label_col: "label"}))

    train_ds = train_ds.map(tokenize_fn, batched=True)
    val_ds   = val_ds.map(tokenize_fn, batched=True)
    test_ds  = test_ds.map(tokenize_fn, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    ).to(DEVICE)

    # Class weights for imbalanced tasks
    from sklearn.utils.class_weight import compute_class_weight
    cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_df[label_col].values)
    class_weights = torch.tensor(cw, dtype=torch.float).to(DEVICE)
    print(f"Class weights: {class_weights}")

    # Custom trainer with class weights
    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            logits = outputs.logits
            loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)
            loss = loss_fct(logits, labels)
            return (loss, outputs) if return_outputs else loss

    output_dir = f"./results/distilbert_{task_name.lower()}_seed{seed}"

    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        seed=seed,
        fp16=False,
        report_to="none",
        logging_steps=50,
        disable_tqdm=True,
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[],
    )

    trainer.train()

    # Predict on test set (avoids NotebookProgressCallback issues with evaluate)
    predictions = trainer.predict(test_ds)
    y_pred = np.argmax(predictions.predictions, axis=-1)
    y_true = predictions.label_ids

    macro_f1 = f1_score(y_true, y_pred, average="macro")
    precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_true, y_pred, average="macro", zero_division=0)

    print(f"\nTest results for {task_name}:")
    print(f"  Macro-F1:   {macro_f1:.4f}")
    print(f"  Precision:  {precision:.4f}")
    print(f"  Recall:     {recall:.4f}")
    print(f"\nClassification Report ({task_name}):")
    print(classification_report(y_true, y_pred, digits=4))

    return {
        "model": "DistilBERT",
        "task": task_name,
        "seed": seed,
        "macro_f1": macro_f1,
        "precision": precision,
        "recall": recall,
        "y_pred": y_pred,
        "y_true": y_true,
        "trainer": trainer,
        "model_obj": model,
    }

print("Training function defined.")

Training function defined.


In [6]:
# --- Sentiment, seed=42 ---
sent_results_1 = train_transformer("Sentiment", "Sentiment", seed=42)


############################################################
# Training DistilBERT for: Sentiment (seed=42)
############################################################


Map:   0%|          | 0/3747 [00:00<?, ? examples/s]

Map: 100%|██████████| 3747/3747 [00:00<00:00, 32275.54 examples/s]

Map: 100%|██████████| 3747/3747 [00:00<00:00, 31763.41 examples/s]

Map:   0%|          | 0/313 [00:00<?, ? examples/s]

...

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.24it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].


There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


{'train_runtime': '360.6', 'train_samples_per_second': '31.17', 'train_steps_per_second': '1.955', 'train_loss': '0.2593', 'epoch': '3'}



Test results for Sentiment:
  Macro-F1:   0.8827
  Precision:  0.8827
  Recall:     0.8826

Classification Report (Sentiment):
              precision    recall  f1-score   support

           0     0.8833    0.8881    0.8857      1117
           1     0.8821    0.8771    0.8796      1066

    accuracy                         0.8827      2183
   macro avg     0.8827    0.8826    0.8827      2183
weighted avg     0.8827    0.8827    0.8827      2183



In [7]:
# --- Sarcasm, seed=42 ---
sarc_results_1 = train_transformer("Sarcasm", "Sarcasm", seed=42)


############################################################
# Training DistilBERT for: Sarcasm (seed=42)
############################################################


Map:   0%|          | 0/3747 [00:00<?, ? examples/s]

Map:  27%|██▋       | 1000/3747 [00:00<00:00, 9527.01 examples/s]

Map: 100%|██████████| 3747/3747 [00:00<00:00, 20185.44 examples/s]

Map:   0%|          | 0/313 [00:00<?, ? examples/s]

...

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.70it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].


There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


{'train_runtime': '309.9', 'train_samples_per_second': '36.27', 'train_steps_per_second': '2.275', 'train_loss': '0.4126', 'epoch': '3'}



Test results for Sarcasm:
  Macro-F1:   0.6782
  Precision:  0.6636
  Recall:     0.7005

Classification Report (Sarcasm):
              precision    recall  f1-score   support

           0     0.9190    0.8765    0.8972      1878
           1     0.4082    0.5246    0.4591       305

    accuracy                         0.8273      2183
   macro avg     0.6636    0.7005    0.6782      2183
weighted avg     0.8477    0.8273    0.8360      2183



In [8]:
# --- Sentiment, seed=123 (second run for stability) ---
sent_results_2 = train_transformer("Sentiment", "Sentiment", seed=123)


############################################################
# Training DistilBERT for: Sentiment (seed=123)
############################################################


Map:   0%|          | 0/3747 [00:00<?, ? examples/s]

Map: 100%|██████████| 3747/3747 [00:00<00:00, 32397.83 examples/s]

Map: 100%|██████████| 3747/3747 [00:00<00:00, 31927.37 examples/s]

Map:   0%|          | 0/313 [00:00<?, ? examples/s]

...

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.72it/s]

{'train_runtime': '325.2', 'train_samples_per_second': '34.57', 'train_steps_per_second': '2.168', 'train_loss': '0.2538', 'epoch': '3'}


There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].


There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Test results for Sentiment:
  Macro-F1:   0.8767
  Precision:  0.8767
  Recall:     0.8767

Classification Report (Sentiment):
              precision    recall  f1-score   support

           0     0.8806    0.8782    0.8794      1117
           1     0.8728    0.8752    0.8740      1066

    accuracy                         0.8768      2183
   macro avg     0.8767    0.8767    0.8767      2183
weighted avg     0.8768    0.8768    0.8768      2183



In [9]:
# --- Sarcasm, seed=123 (second run for stability) ---
sarc_results_2 = train_transformer("Sarcasm", "Sarcasm", seed=123)


############################################################
# Training DistilBERT for: Sarcasm (seed=123)
############################################################


Map:   0%|          | 0/3747 [00:00<?, ? examples/s]

Map:  80%|████████  | 3000/3747 [00:00<00:00, 26057.03 examples/s]

Map: 100%|██████████| 3747/3747 [00:00<00:00, 27306.11 examples/s]

Map:   0%|          | 0/313 [00:00<?, ? examples/s]

...

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.76it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].


There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


{'train_runtime': '316.6', 'train_samples_per_second': '35.5', 'train_steps_per_second': '2.226', 'train_loss': '0.4233', 'epoch': '3'}



Test results for Sarcasm:
  Macro-F1:   0.6674
  Precision:  0.6485
  Recall:     0.7367

Classification Report (Sarcasm):
              precision    recall  f1-score   support

           0     0.9370    0.8078    0.8676      1878
           1     0.3599    0.6656    0.4672       305

    accuracy                         0.7879      2183
   macro avg     0.6485    0.7367    0.6674      2183
weighted avg     0.8564    0.7879    0.8117      2183



## 4 — Comparison Table & Confusion Matrices

In [10]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 150})

# Build comparison table
rows = []
for r in baseline_results:
    rows.append({
        "Model": r["model"],
        "Task": r["task"],
        "Seed": "-",
        "Macro-F1": f"{r['macro_f1']:.4f}",
        "Precision": f"{r['precision']:.4f}",
        "Recall": f"{r['recall']:.4f}",
    })

for r in [sent_results_1, sent_results_2, sarc_results_1, sarc_results_2]:
    rows.append({
        "Model": "DistilBERT",
        "Task": r["task"],
        "Seed": str(r["seed"]),
        "Macro-F1": f"{r['macro_f1']:.4f}",
        "Precision": f"{r['precision']:.4f}",
        "Recall": f"{r['recall']:.4f}",
    })

comparison_df = pd.DataFrame(rows)
print(comparison_df.to_markdown(index=False))
comparison_df

| Model        | Task      | Seed   |   Macro-F1 |   Precision |   Recall |
|:-------------|:----------|:-------|-----------:|------------:|---------:|
| TF-IDF + LR  | Sentiment | -      |     0.8347 |      0.8362 |   0.8344 |
| TF-IDF + SVM | Sentiment | -      |     0.8371 |      0.8378 |   0.8369 |
| TF-IDF + LR  | Sarcasm   | -      |     0.6255 |      0.6129 |   0.6747 |
| TF-IDF + SVM | Sarcasm   | -      |     0.6065 |      0.6055 |   0.6075 |
| DistilBERT   | Sentiment | 42     |     0.8827 |      0.8827 |   0.8826 |
| DistilBERT   | Sentiment | 123    |     0.8767 |      0.8767 |   0.8767 |
| DistilBERT   | Sarcasm   | 42     |     0.6782 |      0.6636 |   0.7005 |
| DistilBERT   | Sarcasm   | 123    |     0.6674 |      0.6485 |   0.7367 |


,Model,Task,Seed,Macro-F1,Precision,Recall
0,TF-IDF + LR,Sentiment,-,0.8347,0.8362,0.8344
1,TF-IDF + SVM,Sentiment,-,0.8371,0.8378,0.8369
2,TF-IDF + LR,Sarcasm,-,0.6255,0.6129,0.6747
3,TF-IDF + SVM,Sarcasm,-,0.6065,0.6055,0.6075
4,DistilBERT,Sentiment,42,0.8827,0.8827,0.8826
5,DistilBERT,Sentiment,123,0.8767,0.8767,0.8767
6,DistilBERT,Sarcasm,42,0.6782,0.6636,0.7005
7,DistilBERT,Sarcasm,123,0.6674,0.6485,0.7367


In [11]:
# Confusion matrices for best model per task
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

models_to_show = [
    ("TF-IDF + LR", "Sentiment", baseline_results[0]["y_pred"], y_test_sent),
    ("TF-IDF + SVM", "Sentiment", baseline_results[1]["y_pred"], y_test_sent),
    ("DistilBERT", "Sentiment", sent_results_1["y_pred"], sent_results_1["y_true"]),
    ("TF-IDF + LR", "Sarcasm", baseline_results[2]["y_pred"], y_test_sarc),
    ("TF-IDF + SVM", "Sarcasm", baseline_results[3]["y_pred"], y_test_sarc),
    ("DistilBERT", "Sarcasm", sarc_results_1["y_pred"], sarc_results_1["y_true"]),
]

label_names_sent = ["Negative", "Positive"]
label_names_sarc = ["Not Sarcastic", "Sarcastic"]

for idx, (model_name, task, y_pred, y_true) in enumerate(models_to_show):
    row = 0 if task == "Sentiment" else 1
    col = idx % 3 if idx < 3 else idx - 3
    ax = axes[row][col]
    
    label_names = label_names_sent if task == "Sentiment" else label_names_sarc
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=label_names, yticklabels=label_names)
    ax.set_ylabel("Actual")
    ax.set_xlabel("Predicted")
    ax.set_title(f"{model_name} — {task}")

plt.suptitle("Confusion Matrices: Baselines vs DistilBERT", fontweight="bold")
plt.tight_layout()
plt.savefig("q21_confusion_matrices.png", bbox_inches="tight")
plt.show()

In [12]:
# Gap analysis: Transformer improvement over baseline
print("\n" + "="*60)
print("GAP ANALYSIS: DistilBERT vs Best Classical Baseline")
print("="*60)

for task in ["Sentiment", "Sarcasm"]:
    baseline_f1 = max(r["macro_f1"] for r in baseline_results if r["task"] == task)
    baseline_name = next(r["model"] for r in baseline_results if r["task"] == task and r["macro_f1"] == baseline_f1)
    
    if task == "Sentiment":
        distilbert_f1 = sent_results_1["macro_f1"]
    else:
        distilbert_f1 = sarc_results_1["macro_f1"]
    
    gap = distilbert_f1 - baseline_f1
    pct_improvement = (gap / baseline_f1) * 100
    
    print(f"\n{task}:")
    print(f"  Best baseline ({baseline_name}): {baseline_f1:.4f}")
    print(f"  DistilBERT:                      {distilbert_f1:.4f}")
    print(f"  Gap:                             {gap:+.4f} ({pct_improvement:+.1f}%)")

# Stability: compare two seeds
print("\n" + "="*60)
print("STABILITY CHECK: Seed 42 vs Seed 123")
print("="*60)
for name, r1, r2 in [
    ("Sentiment", sent_results_1, sent_results_2),
    ("Sarcasm", sarc_results_1, sarc_results_2),
]:
    diff = abs(r1["macro_f1"] - r2["macro_f1"])
    print(f"{name}: seed42={r1['macro_f1']:.4f}, seed123={r2['macro_f1']:.4f}, diff={diff:.4f}")


GAP ANALYSIS: DistilBERT vs Best Classical Baseline

Sentiment:
  Best baseline (TF-IDF + SVM): 0.8371
  DistilBERT:                      0.8827
  Gap:                             +0.0455 (+5.4%)

Sarcasm:
  Best baseline (TF-IDF + LR): 0.6255
  DistilBERT:                      0.6782
  Gap:                             +0.0527 (+8.4%)

STABILITY CHECK: Seed 42 vs Seed 123
Sentiment: seed42=0.8827, seed123=0.8767, diff=0.0059
Sarcasm: seed42=0.6782, seed123=0.6674, diff=0.0108


## Summary

| Finding | Detail |
|---|---|
| Classical baselines | TF-IDF + LR and TF-IDF + SVM with balanced class weights |
| Transformer | DistilBERT fine-tuned with class-weighted loss (3 epochs, lr=2e-5) |
| Sarcasm class imbalance | Handled via `class_weight='balanced'` (baselines) and `CrossEntropyLoss(weight=...)` (transformer) |
| Stability | Two seeds (42, 123) to confirm results are not due to random chance |
| Metric | Macro-F1 is primary — avoids inflated scores from majority-class prediction |

Saved figure: `q21_confusion_matrices.png`